In [ ]:
# ===================================================================
# CUADERNO 7 · CAPA 3 (modelo PU) · CELDA 0: arranque y auditoría
# Correr después de CADA reinicio del kernel.
# ===================================================================
import os, re, sys, time, gc, glob, json, unicodedata, warnings
import numpy as np, pandas as pd, duckdb
warnings.filterwarnings('ignore')
pd.set_option('display.width', 220); pd.set_option('display.max_columns', 80)

os.chdir(r'C:\SERCOP_FINAL')
R = '03_resultados'; D = '05_documentos'
con = duckdb.connect(); con.execute("PRAGMA threads=8")
print('Carpeta:', os.getcwd(), '| Python', sys.version.split()[0], '| DuckDB', duckdb.__version__)

# ---------- 1) Vistas sobre todo lo que produjimos ----------
FUENTES = {
    'bm3':   'base_maestra_v3.parquet',          # 110 columnas
    'bm4':   'base_maestra_v4.parquet',          # base del modelo
    'ofe':   'oferentes_consolidado.parquet',
    'dir':   'directorio_companias.parquet',
    'rp':    'red_proveedor_anio.parquet',
    're':    'red_entidad_anio.parquet',
    'rc':    'red_cobid_anio.parquet',
    'rm':    'red_comunidad_anio.parquet',
    'rr':    'red_representante_anio.parquet',
    'ix':    'indices_red.parquet',
    'cri_p': 'cri_procesos.parquet',
    'cri_v': 'cri_proveedores.parquet',
    'benf':  'benford_corregido.parquet',
    'cpc':   'cpc_por_ocid.parquet',
    'pares': 'pares_mismo_representante.parquet',
}
print('\n--- Inventario ---')
faltan = []
for v, arch in FUENTES.items():
    ruta = f'{R}/{arch}'
    if not os.path.exists(ruta):
        faltan.append(arch); print(f'  [ ] {v:7s} FALTA  {arch}'); continue
    con.execute(f"CREATE OR REPLACE VIEW {v} AS SELECT * FROM read_parquet('{ruta}')")
    n = con.execute(f"SELECT COUNT(*) FROM {v}").fetchone()[0]
    k = len(con.execute(f"DESCRIBE {v}").fetchall())
    mb = os.path.getsize(ruta)/1e6
    mt = time.strftime('%Y-%m-%d %H:%M', time.localtime(os.path.getmtime(ruta)))
    print(f'  [x] {v:7s} {n:>10,} filas x {k:>3} cols | {mb:7.1f} MB | {mt} | {arch}')
if faltan: print('\n!! FALTAN:', faltan)

# ---------- 2) Vista canónica: excluye RUC vacío y nulo ----------
con.execute("""CREATE OR REPLACE VIEW bmv AS SELECT * FROM bm4
               WHERE proveedor_ruc IS NOT NULL AND TRIM(proveedor_ruc) <> ''""")
print('\nVista bmv (bm4 sin RUC vacío/nulo):',
      f"{con.execute('SELECT COUNT(*) FROM bmv').fetchone()[0]:,} filas")

# ---------- 3) Chequeos de integridad ----------
print('\n--- Chequeos ---')
def chk(nombre, obtenido, esperado, tol=0):
    ok = abs(obtenido - esperado) <= tol
    print(f'  {"OK  " if ok else "FALLA"} {nombre}: {obtenido:,} (esperado {esperado:,})')
    return ok
todo_ok = True
todo_ok &= chk('filas bm3',  con.execute("SELECT COUNT(*) FROM bm3").fetchone()[0], 2736387)
todo_ok &= chk('filas bm4',  con.execute("SELECT COUNT(*) FROM bm4").fetchone()[0], 2736387)
todo_ok &= chk('cols bm3',   len(con.execute("DESCRIBE bm3").fetchall()), 110)
todo_ok &= chk('filas bmv',  con.execute("SELECT COUNT(*) FROM bmv").fetchone()[0], 2690017)
todo_ok &= chk('filas red_proveedor_anio', con.execute("SELECT COUNT(*) FROM rp").fetchone()[0], 559338)
nodo_vacio = con.execute("SELECT COUNT(*) FROM rp WHERE TRIM(COALESCE(proveedor_ruc,'')) = ''").fetchone()[0]
todo_ok &= chk('nodo de RUC vacío en la red (debe ser 0)', nodo_vacio, 0)
cap_med = con.execute("""SELECT ROUND(MEDIAN(k),0) FROM (
    SELECT DISTINCT proveedor_ruc, proveedor_capital_suscrito AS k FROM bmv
    WHERE proveedor_capital_suscrito IS NOT NULL)""").fetchone()[0]
print(f'  {"OK  " if cap_med == 1000 else "REVISAR"} capital mediano por empresa: {cap_med:,.0f} (esperado 1,000)')
fant = con.execute("SELECT COUNT(DISTINCT proveedor_ruc) FROM bmv WHERE proveedor_es_fantasma").fetchone()[0]
print(f'  {"OK  " if fant == 15 else "REVISAR"} RUC fantasma: {fant} (esperado 15)')
print('\n', 'TODO EN ORDEN' if todo_ok else '!! HAY FALLAS: revisar antes de modelar')

# ---------- 4) Utilidades de la Capa 3 ----------
def estratos(df, col_anio='anio_proceso', col_tam='red_n_contratos', n=10):
    """Estrato = año x decil de actividad previa. Para percentiles y permutaciones."""
    t = df[col_tam].fillna(0)
    dec = t.groupby(df[col_anio]).rank(method='first').groupby(df[col_anio]).transform(
        lambda s: pd.qcut(s, n, labels=False, duplicates='drop'))
    return df[col_anio].astype(str) + '_' + dec.astype(str)

def pct_intra(df, cols, estrato):
    """Percentil de cada variable dentro del estrato (comparable entre años)."""
    return pd.DataFrame({c: df.groupby(estrato)[c].rank(pct=True) for c in cols})

def norm_nombre(s):
    if pd.isna(s): return None
    s = unicodedata.normalize('NFKD', str(s)).encode('ascii','ignore').decode('ascii').upper()
    s = re.sub(r'[^A-Z ]',' ',s); s = re.sub(r'\s+',' ',s).strip()
    return s if len(s) >= 8 and len(s.split()) >= 2 else None

print('\nUtilidades cargadas: estratos(), pct_intra(), norm_nombre()')
print('\nREGLAS FIJAS DEL PROYECTO')
print('  1. Todo análisis por proveedor usa bmv, nunca bm3/bm4 directo (RUC vacío = $2.338M falsos).')
print('  2. Cero fuga temporal: las variables de red son de años ANTERIORES (anio_aplicable).')
print('  3. Nada de imputar: los faltantes van como bandera (sin_historial_*).')
print('  4. Los fantasmas del SRI solo validan y documentan; 15 RUC no dan para entrenar solo con eso.')
print('  5. Percentil dentro del año: la cobertura de oferentes cambia mucho entre años.')
print('  6. No volver a correr la Celda 2 del cuaderno 6, ni la 13 sin el filtro de RUC vacío.')

In [ ]:
# ===================================================================
# CUADERNO 7 · CELDA 1b — Etiqueta de fantasma CON FECHA (versión corregida)
# Reemplaza a la Celda 1: usa los nombres reales de las columnas del catastro.
# ===================================================================
import os, glob, warnings
import numpy as np, pandas as pd, duckdb
warnings.filterwarnings('ignore')
R = '03_resultados'; D = '05_documentos'

cand = glob.glob(f'{D}/*antasma*.csv') + glob.glob(f'{D}/**/*antasma*.csv', recursive=True)
ARCH = cand[0]; print('Catastro:', ARCH)
fx = pd.read_csv(ARCH, sep='|', encoding='latin-1', dtype=str)
fx.columns = [c.strip() for c in fx.columns]            # el archivo trae ' RAZON_SOCIAL' con espacio
print(f'Filas: {len(fx):,}')

def fecha_excel(s):
    n = pd.to_numeric(s, errors='coerce')
    f = pd.to_datetime(n, unit='D', origin='1899-12-30', errors='coerce')
    return f.fillna(pd.to_datetime(s, errors='coerce', dayfirst=True))

# Diagnóstico: TODAS las columnas que parecen fecha, con su tasa de llenado
print('\n--- Columnas de fecha del catastro ---')
fechas = [c for c in fx.columns if c.upper().startswith('FECHA')]
for c in fechas:
    v = fecha_excel(fx[c])
    r = f'{v.min():%Y-%m-%d} a {v.max():%Y-%m-%d}' if v.notna().any() else 'sin datos'
    print(f'  {c:35s} {v.notna().sum():>5,} con fecha  | {r}')

MAPA = {                      # nombre interno -> nombre exacto en el archivo
 'f_oficio'      : 'FECHA_NOTIFICACION_OFICIO',
 'f_resolucion'  : 'FECHA_NOTIFICACION_RESOLUCION',
 'f_inicio_cal'  : 'FECHA_INICIO_CALIFICACION',
 'f_fin_cal'     : 'FECHA_FIN_CALIFICACION',
 'f_baja'        : 'FECHA_RESOLUCION_SENTENCIA_BAJA',
 'f_react_notif' : 'FECHA_NOTIFICACION_REACTIVACION',
 'f_react_ruc'   : 'FECHA_REACTIVACION_RUC',
}
for k, c in MAPA.items():
    fx[k] = fecha_excel(fx[c]) if c in fx.columns else pd.NaT
    if c not in fx.columns: print(f'  !! no existe la columna {c}')

fx['ruc'] = fx['NUMERO_RUC'].str.replace(r'\.0$','',regex=True).str.strip().str.zfill(13)
fx['f_reactivacion'] = fx[['f_react_notif','f_react_ruc']].min(axis=1)

for c in ['ESTADO_RUC','ACEPTA_RECHAZA','ESTADO_IMPUGNACIÓN','DETALLE_CALIFICACION']:
    if c in fx.columns:
        print(f'\n{c}:'); print(fx[c].fillna('(vacío)').value_counts().head(6).to_string())

sri = (fx.groupby('ruc')
         .agg(f_oficio=('f_oficio','min'), f_resolucion=('f_resolucion','min'),
              f_inicio_cal=('f_inicio_cal','min'), f_fin_cal=('f_fin_cal','max'),
              f_reactivacion=('f_reactivacion','max'), f_baja=('f_baja','min'))
         .reset_index())
sri['f_marca'] = sri[['f_oficio','f_resolucion','f_inicio_cal']].min(axis=1)
print(f'\nRUC únicos: {len(sri):,} | con reactivación: {sri.f_reactivacion.notna().sum():,} '
      f'| con fin de calificación: {sri.f_fin_cal.notna().sum():,}')

# ---------- Etiqueta ----------
cols = [c[0] for c in con.execute("DESCRIBE bmv").fetchall()]
c_fecha = next((c for c in ['fecha_adjudicacion','fecha_publicacion','fecha_release',
                            'fecha_release_json','fecha'] if c in cols), None)
fecha_sql = (f'TRY_CAST("{c_fecha}" AS TIMESTAMP)' if c_fecha else 'NULL')
print('\nFecha del proceso tomada de:', c_fecha)

con.register('sri_t', sri[['ruc','f_marca','f_fin_cal','f_reactivacion']])
con.execute(f"""
CREATE OR REPLACE VIEW etiqueta AS
WITH p AS (SELECT ocid, proveedor_ruc, anio_proceso,
                  COALESCE({fecha_sql}, MAKE_TIMESTAMP(anio_proceso,12,31,0,0,0)) AS fecha_proc
           FROM bmv)
SELECT p.ocid, p.proveedor_ruc, p.anio_proceso, p.fecha_proc,
       (s.ruc IS NOT NULL) AS en_catastro,
       s.f_marca, s.f_fin_cal, s.f_reactivacion,
       (s.ruc IS NOT NULL
        AND (s.f_reactivacion IS NULL OR p.fecha_proc < s.f_reactivacion)) AS y_fantasma,
       CASE WHEN s.ruc IS NULL THEN NULL
            WHEN s.f_marca IS NOT NULL AND p.fecha_proc <  s.f_marca THEN 'antes de la marca'
            WHEN s.f_reactivacion IS NOT NULL AND p.fecha_proc >= s.f_reactivacion THEN 'tras reactivacion'
            ELSE 'durante la calificacion' END AS momento
FROM p LEFT JOIN sri_t s ON s.ruc = p.proveedor_ruc""")

print('\n--- Efecto de la corrección ---')
print(con.execute("""
SELECT COUNT(DISTINCT CASE WHEN en_catastro THEN proveedor_ruc END) AS ruc_en_catastro,
       COUNT(DISTINCT CASE WHEN y_fantasma  THEN proveedor_ruc END) AS ruc_positivos,
       SUM(CASE WHEN en_catastro THEN 1 ELSE 0 END) AS filas_catastro,
       SUM(CASE WHEN y_fantasma  THEN 1 ELSE 0 END) AS filas_positivas
FROM etiqueta""").df().to_string(index=False))
print('\nMomento de los procesos de empresas del catastro:')
print(con.execute("""SELECT momento, COUNT(*) AS filas, COUNT(DISTINCT proveedor_ruc) AS ruc
                     FROM etiqueta WHERE en_catastro GROUP BY 1 ORDER BY 2 DESC""").df().to_string(index=False))
ex = con.execute("""SELECT proveedor_ruc, COUNT(*) AS excluidas, MIN(anio_proceso) AS desde,
                           MAX(anio_proceso) AS hasta, MAX(f_reactivacion) AS reactivada
                    FROM etiqueta WHERE en_catastro AND NOT y_fantasma GROUP BY 1 ORDER BY 2 DESC""").df()
print(f'\nProcesos excluidos por reactivación: {int(ex.excluidas.sum()) if len(ex) else 0}')
if len(ex): print(ex.to_string(index=False))

con.execute(f"""COPY (SELECT ocid, proveedor_ruc, anio_proceso, fecha_proc, en_catastro,
                             f_marca, f_fin_cal, f_reactivacion, y_fantasma, momento FROM etiqueta)
                TO '{R}/etiqueta_fantasma.tmp.parquet' (FORMAT PARQUET, COMPRESSION ZSTD)""")
os.replace(f'{R}/etiqueta_fantasma.tmp.parquet', f'{R}/etiqueta_fantasma.parquet')
con.execute(f"CREATE OR REPLACE VIEW etq AS SELECT * FROM read_parquet('{R}/etiqueta_fantasma.parquet')")
print('\nGuardado etiqueta_fantasma.parquet (reemplaza al de la Celda 1)')

In [ ]:
# ===================================================================
# CUADERNO 7 · CELDA 2 — CRI v2: índice de riesgo SIN Benford
# Línea base contra la que se medirá el modelo de la Capa 3.
# ===================================================================
import os, warnings
import numpy as np, pandas as pd, duckdb
warnings.filterwarnings('ignore')
R = '03_resultados'
cols = set(c[0] for c in con.execute("DESCRIBE bmv").fetchall())

# ---------- 1) Qué señales están disponibles ----------
CAND = {
 'competencia': {
   'c_unico':        ('CASE WHEN tiene_periodo_competitivo THEN (CASE WHEN proveedor_unico THEN 1.0 ELSE 0.0 END) END', 'bin',
                      ['tiene_periodo_competitivo','proveedor_unico']),
   'c_conc_rival':   ('concentracion_rival_habitual', 'pct', ['concentracion_rival_habitual']),
   'c_dominancia':   ('dominancia_maxima_historica',  'pct', ['dominancia_maxima_historica']),
   'c_winrate':      ('win_rate',                     'pct', ['win_rate']),
 },
 'capacidad': {
   'k_ratio_capital':('ratio_monto_capital',          'pct', ['ratio_monto_capital']),
   'k_nuevo':        ('CASE WHEN proveedor_es_nuevo_en_mercado THEN 1.0 ELSE 0.0 END', 'bin',
                      ['proveedor_es_nuevo_en_mercado']),
   'k_cap_minimo':   ('CASE WHEN proveedor_capital_suscrito IS NULL THEN NULL '
                      'WHEN proveedor_capital_suscrito <= 800 THEN 1.0 ELSE 0.0 END', 'bin',
                      ['proveedor_capital_suscrito']),
   'k_joven':        ('-1.0 * anios_desde_primera_actividad_general', 'pct',
                      ['anios_desde_primera_actividad_general']),
 },
 'montos': {
   'm_vs_cpc':       ('CASE WHEN med_cpc > 0 THEN monto_final / med_cpc END',  'pct', ['cpc5_principal','monto_final']),
   'm_vs_tipo':      ('CASE WHEN med_tipo > 0 THEN monto_final / med_tipo END','pct', ['tipo_contratacion','monto_final']),
   'm_redondo':      ('CASE WHEN monto_final IS NULL OR monto_final <= 0 THEN NULL '
                      'WHEN ABS(monto_final - ROUND(monto_final/1000.0)*1000.0) < 0.005 THEN 1.0 ELSE 0.0 END','bin',
                      ['monto_final']),
 },
}
usar = {}
print('--- Señales disponibles ---')
for dim, d in CAND.items():
    usar[dim] = {}
    for nombre, (expr, tipo, req) in d.items():
        ok = all(c in cols for c in req)
        print(f'  {"[x]" if ok else "[ ]"} {dim:12s} {nombre:16s} requiere {req}')
        if ok: usar[dim][nombre] = (expr, tipo)
    assert usar[dim], f'La dimensión {dim} se quedó sin señales'
print({k: list(v) for k, v in usar.items()})

# ---------- 2) Construir el índice en DuckDB ----------
def pctl(e):   # percentil entre los NO nulos del mismo año
    return (f'CASE WHEN ({e}) IS NULL THEN NULL ELSE '
            f'(RANK() OVER (PARTITION BY anio_proceso ORDER BY ({e})) - 1.0) / '
            f'NULLIF(COUNT({e}) OVER (PARTITION BY anio_proceso) - 1.0, 0) END')

sel, nombres = [], []
for dim, d in usar.items():
    for nombre, (expr, tipo) in d.items():
        sel.append(f'{pctl(expr) if tipo == "pct" else f"({expr})"} AS {nombre}')
        nombres.append((dim, nombre))

med_cpc  = ('MEDIAN(monto_final) OVER (PARTITION BY cpc5_principal, anio_proceso)'
            if 'cpc5_principal' in cols else 'NULL')
med_tipo = ('MEDIAN(monto_final) OVER (PARTITION BY tipo_contratacion, anio_proceso)'
            if 'tipo_contratacion' in cols else 'NULL')

def media_disp(cs):   # promedio de lo disponible, sin imputar
    num = ' + '.join([f'COALESCE({c},0)' for c in cs])
    den = ' + '.join([f'CASE WHEN {c} IS NULL THEN 0 ELSE 1 END' for c in cs])
    return f'CASE WHEN ({den}) = 0 THEN NULL ELSE ({num}) / ({den}) END'

dims = {dim: media_disp(list(d)) for dim, d in usar.items()}
cri  = media_disp([f'dim_{k}' for k in dims])

sql = f"""
COPY (
  WITH b AS (
    SELECT ocid, proveedor_ruc, anio_proceso, monto_final,
           {med_cpc}  AS med_cpc,
           {med_tipo} AS med_tipo,
           * EXCLUDE (ocid, proveedor_ruc, anio_proceso, monto_final)
    FROM bmv
  ),
  s AS (SELECT ocid, proveedor_ruc, anio_proceso, {', '.join(sel)} FROM b),
  d AS (SELECT *, {', '.join([f'{v} AS dim_{k}' for k, v in dims.items()])} FROM s)
  SELECT *, {cri} AS cri2
  FROM d
) TO '{R}/cri_v2_procesos.tmp.parquet' (FORMAT PARQUET, COMPRESSION ZSTD)
"""
import time; t0 = time.time(); con.execute(sql); print(f'\nCRI v2 calculado en {time.time()-t0:.0f} s')
os.replace(f'{R}/cri_v2_procesos.tmp.parquet', f'{R}/cri_v2_procesos.parquet')
con.execute(f"CREATE OR REPLACE VIEW cri2 AS SELECT * FROM read_parquet('{R}/cri_v2_procesos.parquet')")
print('filas:', f"{con.execute('SELECT COUNT(*) FROM cri2').fetchone()[0]:,}")

# ---------- 3) Diagnóstico ----------
print('\nCobertura de cada dimensión y del índice:')
print(con.execute("""
SELECT ROUND(100.0*AVG(CASE WHEN dim_competencia IS NOT NULL THEN 1 ELSE 0 END),1) AS competencia,
       ROUND(100.0*AVG(CASE WHEN dim_capacidad   IS NOT NULL THEN 1 ELSE 0 END),1) AS capacidad,
       ROUND(100.0*AVG(CASE WHEN dim_montos      IS NOT NULL THEN 1 ELSE 0 END),1) AS montos,
       ROUND(100.0*AVG(CASE WHEN cri2            IS NOT NULL THEN 1 ELSE 0 END),1) AS cri2
FROM cri2""").df().to_string(index=False))
print('\nCorrelación entre dimensiones (deben ser bajas: miden cosas distintas):')
print(con.execute("""SELECT ROUND(CORR(dim_competencia,dim_capacidad),3) AS comp_cap,
                            ROUND(CORR(dim_competencia,dim_montos),3)   AS comp_mon,
                            ROUND(CORR(dim_capacidad,dim_montos),3)     AS cap_mon FROM cri2""").df().to_string(index=False))
print('\nDistribución del CRI v2:')
print(con.execute("""SELECT ROUND(AVG(cri2),3) AS media, ROUND(MEDIAN(cri2),3) AS mediana,
                            ROUND(QUANTILE_CONT(cri2,0.9),3) AS p90, ROUND(QUANTILE_CONT(cri2,0.99),3) AS p99
                     FROM cri2""").df().to_string(index=False))

# ---------- 4) Agregado por proveedor-año (lo que usará el modelo) ----------
con.execute(f"""COPY (
  SELECT proveedor_ruc, anio_proceso,
         COUNT(*) AS n_procesos,
         AVG(cri2) AS cri2_medio, MAX(cri2) AS cri2_max,
         AVG(dim_competencia) AS dim_competencia, AVG(dim_capacidad) AS dim_capacidad,
         AVG(dim_montos) AS dim_montos
  FROM cri2 GROUP BY 1,2
) TO '{R}/cri_v2_proveedor_anio.tmp.parquet' (FORMAT PARQUET, COMPRESSION ZSTD)""")
os.replace(f'{R}/cri_v2_proveedor_anio.tmp.parquet', f'{R}/cri_v2_proveedor_anio.parquet')
con.execute(f"CREATE OR REPLACE VIEW cri2v AS SELECT * FROM read_parquet('{R}/cri_v2_proveedor_anio.parquet')")
print('\nAgregado proveedor-año:', f"{con.execute('SELECT COUNT(*) FROM cri2v').fetchone()[0]:,} filas")

# ---------- 5) Dónde caen los positivos ----------
d = con.execute("""
SELECT v.proveedor_ruc, v.anio_proceso, v.cri2_medio, v.n_procesos,
       MAX(CASE WHEN e.y_fantasma THEN 1 ELSE 0 END) AS y
FROM cri2v v LEFT JOIN etq e
  ON e.proveedor_ruc = v.proveedor_ruc AND e.anio_proceso = v.anio_proceso
GROUP BY 1,2,3,4""").df()
d['pct'] = d.groupby('anio_proceso')['cri2_medio'].rank(pct=True)
print(f"\nPositivos en el panel proveedor-año: {int(d.y.sum())} filas, {d.loc[d.y==1,'proveedor_ruc'].nunique()} RUC")
if d.y.sum():
    print(f"  percentil mediano del CRI v2 -> positivos {d.loc[d.y==1,'pct'].median():.3f} | resto {d.loc[d.y==0,'pct'].median():.3f}")
    print(f"  positivos en el decil superior: {int((d.loc[d.y==1,'pct']>0.9).sum())} de {int(d.y.sum())}")
print('\nGuardado: cri_v2_procesos.parquet y cri_v2_proveedor_anio.parquet')

In [ ]:
# ===================================================================
# CUADERNO 7 · CELDA 3 — Panel de modelado (proveedor x año) + 2 etiquetas
# ===================================================================
import os, time, warnings
import numpy as np, pandas as pd, duckdb
warnings.filterwarnings('ignore')
R = '03_resultados'
PREF_PA = ('red_','cob_','com_','rep_','idx_','grupo_','n_comp_')  # constantes dentro del proveedor-año
# ent_* describe a la ENTIDAD compradora: varía entre procesos del mismo proveedor, se promedia

# ---------- 1) Clasificar columnas ----------
desc = con.execute("DESCRIBE bmv").fetchall()
num  = [c for c, t, *_ in desc if any(k in t.upper() for k in ('INT','DOUBLE','FLOAT','DECIMAL','BIGINT'))]
boo  = [c for c, t, *_ in desc if 'BOOLEAN' in t.upper()]
llaves = {'anio_proceso','ocid','proveedor_ruc','entidad_ruc'}
cte  = [c for c in num + boo if c.startswith(PREF_PA) and c not in llaves]
var  = [c for c in num + boo if not c.startswith(PREF_PA) and c not in llaves
        and not c.endswith('_pct')]
print(f'Columnas constantes por proveedor-año: {len(cte)} | que varían por proceso: {len(var)}')

mue = cte[:6] + [c for c in var if c.startswith('ent_')][:2]
chk = con.execute(f"""
SELECT {', '.join([f'SUM(CASE WHEN n_{i} > 1 THEN 1 ELSE 0 END) AS "{c}"' for i, c in enumerate(mue)])}
FROM (SELECT proveedor_ruc, anio_proceso,
             {', '.join([f'COUNT(DISTINCT "{c}") AS n_{i}' for i, c in enumerate(mue)])}
      FROM bmv GROUP BY 1,2)""").df()
print('Grupos con más de un valor distinto (0 en las red_/cob_/com_; las ent_ SÍ varían, por eso se promedian):')
print(chk.T.rename(columns={0:'grupos'}).to_string())

# ---------- 2) Etiqueta B: competencia intragrupo contemporánea ----------
dr = con.execute(f"SELECT RUC, REPRESENTANTE FROM read_parquet('{R}/directorio_companias.parquet')").df().drop_duplicates('RUC')
dr['rep_norm'] = dr.REPRESENTANTE.map(norm_nombre)
dr = dr[dr.rep_norm.notna()]
tam = dr.groupby('rep_norm')['RUC'].size()
dr = dr[dr.rep_norm.map(tam).between(2, 30)]          # mismo criterio de la Capa 2
of = con.execute("""SELECT DISTINCT ocid, oferente_ruc AS ruc FROM ofe
                    WHERE TRIM(COALESCE(oferente_ruc,'')) <> ''""").df()
t = of.groupby('ocid')['ruc'].transform('size'); of = of[(t>=2)&(t<=40)]
og = of.merge(dr[['RUC','rep_norm']], left_on='ruc', right_on='RUC', how='inner')
cnt = og.groupby(['ocid','rep_norm'])['ruc'].nunique()
casos = cnt[cnt>=2].reset_index()[['ocid','rep_norm']]
print(f'\nProcesos con competencia intragrupo: {casos.ocid.nunique():,}')
con.register('casos_t', casos); con.register('dr_t', dr[['RUC','rep_norm']])

# ---------- 3) Armar el panel ----------
agg = (['COUNT(*) AS n_procesos',
        'COUNT(DISTINCT entidad_ruc) AS n_entidades_anio',
        'SUM(monto_final) AS monto_anio',
        'MAX(monto_final) AS monto_max_anio']
       + [f'AVG(CAST("{c}" AS DOUBLE)) AS "{c}"' for c in var]
       + [f'MAX("{c}") AS "{c}"' for c in cte])
sql = f"""
COPY (
  WITH pa AS (
    SELECT proveedor_ruc, anio_proceso, {', '.join(agg)}
    FROM bmv GROUP BY 1,2
  ),
  cri AS (SELECT proveedor_ruc, anio_proceso, cri2_medio, cri2_max,
                 dim_competencia, dim_capacidad, dim_montos
          FROM read_parquet('{R}/cri_v2_proveedor_anio.parquet')),
  ya AS (SELECT proveedor_ruc, anio_proceso,
                MAX(CASE WHEN y_fantasma THEN 1 ELSE 0 END) AS y_fantasma,
                MAX(CASE WHEN en_catastro THEN 1 ELSE 0 END) AS en_catastro
         FROM read_parquet('{R}/etiqueta_fantasma.parquet') GROUP BY 1,2),
  yb AS (SELECT b.proveedor_ruc, b.anio_proceso,
                MAX(CASE WHEN c.ocid IS NOT NULL THEN 1 ELSE 0 END) AS y_intragrupo,
                SUM(CASE WHEN c.ocid IS NOT NULL THEN 1 ELSE 0 END) AS n_proc_intragrupo
         FROM bmv b
         JOIN dr_t d ON d.RUC = b.proveedor_ruc
         LEFT JOIN casos_t c ON c.ocid = b.ocid AND c.rep_norm = d.rep_norm
         GROUP BY 1,2)
  SELECT pa.*,
         cri.cri2_medio, cri.cri2_max, cri.dim_competencia, cri.dim_capacidad, cri.dim_montos,
         COALESCE(ya.y_fantasma,0)   AS y_fantasma,
         COALESCE(ya.en_catastro,0)  AS en_catastro,
         COALESCE(yb.y_intragrupo,0) AS y_intragrupo,
         COALESCE(yb.n_proc_intragrupo,0) AS n_proc_intragrupo,
         (yb.proveedor_ruc IS NOT NULL) AS tiene_grupo
  FROM pa
  LEFT JOIN cri ON cri.proveedor_ruc=pa.proveedor_ruc AND cri.anio_proceso=pa.anio_proceso
  LEFT JOIN ya  ON ya.proveedor_ruc =pa.proveedor_ruc AND ya.anio_proceso =pa.anio_proceso
  LEFT JOIN yb  ON yb.proveedor_ruc =pa.proveedor_ruc AND yb.anio_proceso =pa.anio_proceso
) TO '{R}/panel_modelo.tmp.parquet' (FORMAT PARQUET, COMPRESSION ZSTD)
"""
t0 = time.time(); con.execute(sql); print(f'\nPanel construido en {time.time()-t0:.0f} s')
os.replace(f'{R}/panel_modelo.tmp.parquet', f'{R}/panel_modelo.parquet')
con.execute(f"CREATE OR REPLACE VIEW panel AS SELECT * FROM read_parquet('{R}/panel_modelo.parquet')")

n = con.execute("SELECT COUNT(*) FROM panel").fetchone()[0]
k = len(con.execute("DESCRIBE panel").fetchall())
print(f'panel_modelo: {n:,} filas x {k} columnas')

# ---------- 4) Las dos etiquetas ----------
print('\n--- Objetivo A: empresa fantasma ---')
print(con.execute("""SELECT COUNT(*) AS filas_pos, COUNT(DISTINCT proveedor_ruc) AS ruc_pos,
                            MIN(anio_proceso) AS desde, MAX(anio_proceso) AS hasta
                     FROM panel WHERE y_fantasma=1""").df().to_string(index=False))
print('\n--- Objetivo B: competencia intragrupo ---')
print(con.execute("""SELECT COUNT(*) AS filas_pos, COUNT(DISTINCT proveedor_ruc) AS ruc_pos,
                            SUM(n_proc_intragrupo) AS procesos
                     FROM panel WHERE y_intragrupo=1""").df().to_string(index=False))
print('\nProveedores con grupo identificable (universo del objetivo B):',
      f"{con.execute('SELECT COUNT(*) FROM panel WHERE tiene_grupo').fetchone()[0]:,} filas")
print('\nSolapamiento entre objetivos:')
print(con.execute("""SELECT y_fantasma, y_intragrupo, COUNT(*) AS filas
                     FROM panel GROUP BY 1,2 ORDER BY 1 DESC, 2 DESC""").df().to_string(index=False))
print('\nPositivos del objetivo B por año:')
print(con.execute("""SELECT anio_proceso AS anio, COUNT(*) AS filas_pos,
                            COUNT(DISTINCT proveedor_ruc) AS ruc
                     FROM panel WHERE y_intragrupo=1 GROUP BY 1 ORDER BY 1""").df().to_string(index=False))

# ---------- 5) Familias de variables (para excluir según el objetivo) ----------
pc = [c[0] for c in con.execute("DESCRIBE panel").fetchall()]
fam = {'red':[c for c in pc if c.startswith('red_')], 'cob':[c for c in pc if c.startswith('cob_')],
       'com':[c for c in pc if c.startswith('com_')], 'rep':[c for c in pc if c.startswith('rep_') or c.startswith('grupo_')],
       'ent':[c for c in pc if c.startswith('ent_')], 'idx':[c for c in pc if c.startswith('idx_')],
       'cri':[c for c in pc if c.startswith('cri2') or c.startswith('dim_')]}
fam['base'] = [c for c in pc if c not in sum(fam.values(), []) and c not in
               ('proveedor_ruc','anio_proceso','y_fantasma','y_intragrupo','en_catastro',
                'n_proc_intragrupo','tiene_grupo')]
print('\n--- Familias de variables ---')
for f, cs in fam.items(): print(f'  {f:5s} {len(cs):3d}  {cs[:5]}{" ..." if len(cs)>5 else ""}')
print('\nIMPORTANTE: para el objetivo B hay que excluir la familia "rep", que es la versión')
print('rezagada del mismo fenómeno que se quiere predecir.')

In [ ]:
# ===================================================================
# CUADERNO 7 · CELDA 4b — Variables del modelo y AUDITORÍA DE FUGA (corregida)
# Reemplaza a la Celda 4. Regla: la fuga se decide por el ORIGEN de la variable,
# no por su AUC. El AUC solo se reporta para revisión humana.
# ===================================================================
import os, json, warnings
import numpy as np, pandas as pd
warnings.filterwarnings('ignore')
R = '03_resultados'
AVISO_AUC = 0.80        # solo para avisar, no excluye

P = pd.read_parquet(f'{R}/panel_modelo.parquet')
print(f'Panel: {len(P):,} filas x {P.shape[1]} columnas')

LLAVES    = {'proveedor_ruc','anio_proceso'}
ETIQUETAS = {'y_fantasma','y_intragrupo','en_catastro','n_proc_intragrupo','tiene_grupo'}
IDS       = {'com_id'}
# (1) variables que nacen del MISMO registro del que sale la etiqueta A (catastro del SRI):
#     una empresa calificada como fantasma tiene el RUC suspendido por definición.
PAT_SRI  = ('fantasma','estado_riesgoso','suspendid','ruc_estado','catastro')
EXC_SRI  = {c for c in P.columns if any(p in c.lower() for p in PAT_SRI)} - ETIQUETAS
# (2) variables definicionales del objetivo B: ser positivo exige estar en el directorio societario
PAT_DIR  = ('sociedad_registrada','en_directorio','tiene_representante')
EXC_DIR  = {c for c in P.columns if any(p in c.lower() for p in PAT_DIR)}

print('\n--- Exclusiones por origen ---')
print(f'  Siempre (identificador):            {sorted(IDS)}')
print(f'  Objetivo A y B (vienen del SRI):    {sorted(EXC_SRI)}')
print(f'  Objetivo B (definición del grupo):  {sorted(EXC_DIR)}')

num = P.select_dtypes(include=[np.number, bool]).columns
X_cols = [c for c in num if c not in LLAVES | ETIQUETAS | IDS | EXC_SRI]
print(f'\nCandidatas: {len(X_cols)}')

def auc(x, y):
    m = ~pd.isna(x)
    if m.sum() < 20 or y[m].sum() == 0 or y[m].sum() == m.sum(): return np.nan
    r = pd.Series(x[m]).rank().values; yy = y[m]
    n1, n0 = yy.sum(), len(yy) - yy.sum()
    return (r[yy == 1].sum() - n1*(n1+1)/2) / (n1*n0)

aud = pd.DataFrame([(c, auc(P[c].astype(float).values, P.y_fantasma.values),
                        auc(P[c].astype(float).values, P.y_intragrupo.values),
                        float(pd.isna(P[c]).mean())) for c in X_cols],
                   columns=['variable','auc_A','auc_B','pct_nulos'])

lim = AVISO_AUC - 0.5
avisa = aud[(aud.auc_A.sub(0.5).abs() > lim) | (aud.auc_B.sub(0.5).abs() > lim)]
print(f'\n--- AVISO: variables con |AUC-0.5| > {lim:.2f} que SÍ entran al modelo ---')
if len(avisa):
    print(avisa.round(3).to_string(index=False))
    print('Revisa una por una de dónde sale cada una. Si alguna se construyó a partir del')
    print('catastro del SRI o del directorio, agrégala a PAT_SRI / PAT_DIR y vuelve a correr.')
else:
    print('  ninguna')

FAM = lambda pre: [c for c in X_cols if c.startswith(pre)]
fam = {'red':FAM('red_'), 'cob':FAM('cob_'), 'com':FAM('com_'),
       'rep':[c for c in X_cols if c.startswith('rep_') or c.startswith('grupo_')],
       'ent':FAM('ent_'), 'idx':FAM('idx_'),
       'cri':[c for c in X_cols if c.startswith('cri2') or c.startswith('dim_')]}
fam['base'] = [c for c in X_cols if c not in sum(fam.values(), [])]

exc_B = set(fam['rep']) | EXC_DIR
VARS = {'A': X_cols,
        'B': [c for c in X_cols if c not in exc_B]}
ANT = fam['red'] + fam['cob'] + fam['com'] + fam['idx']       # solo años anteriores
VARS['A_anticipada'] = ANT
VARS['B_anticipada'] = [c for c in ANT if c not in exc_B]
alto = set(avisa.variable)                                     # variante estricta, para sensibilidad
VARS['A_estricta'] = [c for c in VARS['A'] if c not in alto]
VARS['B_estricta'] = [c for c in VARS['B'] if c not in alto]
print('\n--- Conjuntos ---')
for k, v in VARS.items(): print(f'  {k:14s} {len(v):3d} variables')

for obj, col in [('A','auc_A'), ('B','auc_B')]:
    s = aud[aud.variable.isin(VARS[obj])]
    print(f'\n--- Más informativas que entran al objetivo {obj} ---')
    print(s.reindex(s[col].sub(0.5).abs().sort_values(ascending=False).index).head(10).round(3).to_string(index=False))

print('\n--- Nulos por familia ---')
for f, cs in fam.items():
    if cs: print(f'  {f:5s} {len(cs):3d} vars | nulos promedio {P[cs].isna().mean().mean()*100:5.1f}%')

with open(f'{R}/vars_modelo.json','w',encoding='utf-8') as f: json.dump(VARS, f, ensure_ascii=False, indent=1)
aud.to_csv(f'{R}/auditoria_fuga.csv', index=False)
print('\nGuardado: vars_modelo.json y auditoria_fuga.csv')

In [ ]:
# ===================================================================
# CUADERNO 7 · CELDA 5 — Modelo PU (bagging) y evaluación honesta
# ===================================================================
import os, re, sys, time, json, unicodedata, warnings
import numpy as np, pandas as pd
warnings.filterwarnings('ignore')
try:
    from sklearn.ensemble import HistGradientBoostingClassifier
except ImportError:
    import subprocess; subprocess.run([sys.executable,'-m','pip','install','scikit-learn'], check=True)
    from sklearn.ensemble import HistGradientBoostingClassifier
R = '03_resultados'

K_BAG     = 100      # modelos por conjunto (bagging PU)

RATIO_U   = 5        # no etiquetados por cada positivo en cada modelo
N_POR_ANIO= 2000     # muestra de población por año para calcular percentiles
B_NULL    = 5000     # repeticiones de la hipótesis nula
SEMILLA   = 20260930
rng = np.random.default_rng(SEMILLA)

assert callable(norm_nombre), 'Falta correr la Celda 0 del cuaderno 7'
P = pd.read_parquet(f'{R}/panel_modelo.parquet')
VARS = json.load(open(f'{R}/vars_modelo.json', encoding='utf-8'))

# ---------- conglomerado del objetivo B: el representante legal, no la empresa ----------
dr = pd.read_parquet(f'{R}/directorio_companias.parquet')[['RUC','REPRESENTANTE']].drop_duplicates('RUC')
dr['rep_norm'] = dr.REPRESENTANTE.map(norm_nombre)
P = P.merge(dr[['RUC','rep_norm']], left_on='proveedor_ruc', right_on='RUC', how='left').drop(columns=['RUC'])
P['grupo_B'] = P.rep_norm.fillna('ruc:' + P.proveedor_ruc)


def pu_bagging(Xp, Xu, Xs, k=K_BAG, ratio=RATIO_U, seed=0):
    """Mordelet y Vert: K modelos, cada uno con todos los positivos y una muestra de no etiquetados."""
    r = np.random.default_rng(seed); acum = np.zeros(len(Xs)); n = len(Xp); hechos = 0
    for i in range(k):
        idx = r.choice(len(Xu), size=min(len(Xu), n*ratio), replace=False)
        Xtr = np.vstack([Xp, Xu[idx]])
        ytr = np.r_[np.ones(n), np.zeros(len(idx))]
        # con pocos positivos, en cada submuestra hay columnas con 0 o 1 valor distinto:
        # el binado de sklearn falla con esas, así que se descartan en cada modelo
        util = np.array([np.unique(Xtr[~np.isnan(Xtr[:, j]), j]).size >= 2
                         for j in range(Xtr.shape[1])])
        if util.sum() == 0: continue
        m = HistGradientBoostingClassifier(max_depth=3, max_iter=60, learning_rate=0.1,
                                           min_samples_leaf=5, l2_regularization=1.0,
                                           random_state=i).fit(Xtr[:, util], ytr)
        acum += m.predict_proba(Xs[:, util])[:, 1]; hechos += 1
    assert hechos > 0, 'Ningún modelo se pudo entrenar: revisar las variables'
    return acum / hechos

def evaluar(objetivo, col_y, col_grupo, conjunto, n_folds=None):
    cols = VARS[conjunto]
    d = P[P[col_y].notna()].copy()
    if objetivo == 'B': d = d[d.tiene_grupo]            # universo: solo quien tiene grupo identificable
    X = d[cols].astype(float).values
    y = d[col_y].values.astype(int); g = d[col_grupo].values
    gp = pd.unique(g[y == 1])
    print(f'\n{"="*62}\nObjetivo {objetivo} | conjunto "{conjunto}" | {len(cols)} variables')
    print(f'  filas: {len(d):,} | positivos: {y.sum()} en {len(gp)} conglomerados')

    anios = d.anio_proceso.values
    idx_pos = np.where(y == 1)[0]
    idx_neg = np.concatenate([rng.choice(np.where((y == 0) & (anios == a))[0],
                              size=min(N_POR_ANIO, ((y == 0) & (anios == a)).sum()), replace=False)
                              for a in np.unique(anios) if ((y == 0) & (anios == a)).sum() > 0])
    idx_s = np.concatenate([idx_pos, idx_neg]); Xs = X[idx_s]
    score = np.full(len(idx_s), np.nan)

    assert len(gp) >= 2, (f'El objetivo {objetivo} tiene {len(gp)} conglomerado(s) de positivos: '
                          'no se puede dejar uno fuera. Revisar la definición del conglomerado.')
    folds = ([[c] for c in gp] if n_folds is None
             else [f for f in np.array_split(rng.permutation(gp), min(n_folds, len(gp))) if len(f)])
    t0 = time.time(); base_pob = np.zeros(len(Xs))
    for j, fuera in enumerate(folds):
        m_tr = (y == 1) & (~pd.Series(g).isin(fuera).values)   # positivos SIN el pliegue
        m_u  = (y == 0)
        s = pu_bagging(X[m_tr], X[m_u], Xs, seed=SEMILLA + j)
        donde = np.isin(g[idx_s], fuera) & (y[idx_s] == 1)
        score[donde] = s[donde]
        if j == 0: base_pob = s                                 # modelo del primer pliegue, para la población
    score[np.isnan(score)] = base_pob[np.isnan(score)]
    print(f'  {len(folds)} pliegues entrenados en {time.time()-t0:.0f} s')

    ev = pd.DataFrame({'g': g[idx_s], 'y': y[idx_s], 'anio': anios[idx_s], 'score': score})
    for nombre, v in [('modelo', score),
                      ('cri_v2', d['cri2_medio'].values[idx_s] if 'cri2_medio' in d else None),
                      ('idx_intermediacion', d['idx_intermediacion'].values[idx_s] if 'idx_intermediacion' in d else None)]:
        if v is None: continue
        ev[nombre] = v
    res = []
    for nombre in [c for c in ['modelo','cri_v2','idx_intermediacion'] if c in ev]:
        pct = ev.groupby('anio')[nombre].rank(pct=True)
        obs = pct[ev.y == 1].median()
        gneg = pd.unique(ev.loc[ev.y == 0, 'g'])
        pos_idx = {gg: np.where(ev.g.values == gg)[0] for gg in gneg}
        tam = len(pd.unique(ev.loc[ev.y == 1, 'g']))
        nul = np.empty(B_NULL); pv = pct.values
        for b in range(B_NULL):
            sel = rng.choice(gneg, size=tam, replace=False)
            nul[b] = np.nanmedian(pv[np.concatenate([pos_idx[s_] for s_ in sel])])
        p = (1 + np.sum(np.abs(nul - 0.5) >= abs(obs - 0.5))) / (B_NULL + 1)
        top10 = float((pct[ev.y == 1] > 0.9).mean()); top25 = float((pct[ev.y == 1] > 0.75).mean())
        res.append((nombre, round(obs,3), round(np.percentile(nul,2.5),3), round(np.percentile(nul,97.5),3),
                    round(p,4), round(top10,3), round(top25,3), round(top10/0.10,1)))
    r = pd.DataFrame(res, columns=['metodo','mediana_percentil','nulo_p2.5','nulo_p97.5','p',
                                   'frac_decil_sup','frac_cuartil_sup','lift_decil'])
    print(r.to_string(index=False))
    return r, ev

rA, evA = evaluar('A', 'y_fantasma',  'proveedor_ruc', 'A')              # dejar una empresa fuera
rB, evB = evaluar('B', 'y_intragrupo','grupo_B',       'B', n_folds=5)   # validación cruzada por grupos
rAa, _  = evaluar('A', 'y_fantasma',  'proveedor_ruc', 'A_anticipada')   # solo variables de años previos
rBa, _  = evaluar('B', 'y_intragrupo','grupo_B',       'B_anticipada', n_folds=5)

out = pd.concat([rA.assign(caso='A completo'), rAa.assign(caso='A anticipada'),
                 rB.assign(caso='B completo'), rBa.assign(caso='B anticipada')])
out.to_csv(f'{R}/resultados_modelo.csv', index=False)
print('\n\n=============== RESUMEN ===============')
print(out[['caso','metodo','mediana_percentil','nulo_p2.5','nulo_p97.5','p','lift_decil']].to_string(index=False))
print('\nGuardado resultados_modelo.csv')
print('lift_decil = cuántas veces más positivos caen en el decil superior que por azar (1.0 = azar)')

In [ ]:
# ===================================================================
# CUADERNO 7 · CELDA 6 — Exploración del directorio de la Superintendencia
# Objetivo: ver qué variables NO estamos usando y si alguna separa a los positivos.
# ===================================================================
import os, re, unicodedata, warnings
import numpy as np, pandas as pd
warnings.filterwarnings('ignore')
R = '03_resultados'

dr = pd.read_parquet(f'{R}/directorio_companias.parquet')
dr.columns = [c.strip() for c in dr.columns]
print(f'Directorio: {len(dr):,} filas x {dr.shape[1]} columnas')
print('Columnas:', list(dr.columns))
dr = dr.drop_duplicates(subset='RUC')

# ---------- 1) Perfil de cada columna ----------
print('\n--- Perfil de columnas ---')
for c in dr.columns:
    nn = dr[c].notna().sum(); card = dr[c].nunique(dropna=True)
    print(f'\n{c}  ({100*nn/len(dr):.1f}% con dato, {card:,} valores distintos)')
    if card <= 25:
        print(dr[c].fillna('(vacío)').value_counts().head(12).to_string())
    else:
        print('  ejemplos:', dr[c].dropna().astype(str).head(3).tolist())

# ---------- 2) Variables derivadas candidatas ----------
def norm_txt(s):
    if pd.isna(s): return None
    return unicodedata.normalize('NFKD', str(s)).encode('ascii','ignore').decode('ascii').upper().strip()

def a_fecha(s):
    n = pd.to_numeric(s, errors='coerce')
    f = pd.to_datetime(n, unit='D', origin='1899-12-30', errors='coerce')
    return f.fillna(pd.to_datetime(s, errors='coerce', dayfirst=True))

col_sit = next((c for c in dr.columns if 'SITUAC' in c.upper()), None)
col_tip = next((c for c in dr.columns if c.upper().strip() == 'TIPO'), None)
col_con = next((c for c in dr.columns if 'CONSTITUC' in c.upper()), None)
col_pro = next((c for c in dr.columns if 'PROVINCIA' in c.upper()), None)
col_car = next((c for c in dr.columns if 'CARGO' in c.upper()), None)
print(f'\nDetectadas -> situación:{col_sit} tipo:{col_tip} constitución:{col_con} provincia:{col_pro} cargo:{col_car}')

sup = pd.DataFrame({'proveedor_ruc': dr.RUC.astype(str)})
if col_sit:
    s = dr[col_sit].map(norm_txt)
    sup['sup_situacion'] = s
    sup['sup_no_activa']   = (~s.fillna('').str.contains('ACTIVA')).astype(float)
    sup['sup_disolucion']  = s.fillna('').str.contains('DISOL|LIQUID|CANCEL').astype(float)
if col_tip: sup['sup_tipo'] = dr[col_tip].map(norm_txt)
if col_con:
    f = a_fecha(dr[col_con]); sup['sup_anio_constitucion'] = f.dt.year
    print(f'\nFecha de constitución: {f.notna().sum():,} con dato | '
          f'{f.min():%Y-%m-%d} a {f.max():%Y-%m-%d}' if f.notna().any() else 'sin fechas')
if col_pro: sup['sup_provincia'] = dr[col_pro].map(norm_txt)
if col_car:
    c2 = dr[col_car].map(norm_txt)
    sup['sup_cargo'] = c2
    sup['sup_rep_liquidador'] = c2.fillna('').str.contains('LIQUIDADOR').astype(float)

P = pd.read_parquet(f'{R}/panel_modelo.parquet')
P = P.merge(sup, on='proveedor_ruc', how='left')
P['sup_antiguedad'] = P.anio_proceso - P.sup_anio_constitucion
P['sup_muy_joven']  = (P.sup_antiguedad <= 1).astype(float).where(P.sup_antiguedad.notna())
P['sup_contrata_antes_de_existir'] = (P.sup_antiguedad < 0).astype(float).where(P.sup_antiguedad.notna())
print(f'\nProveedores del panel con datos del directorio: '
      f'{P.sup_anio_constitucion.notna().mean()*100:.1f}%')
if P.sup_antiguedad.notna().any():
    print('Antigüedad al contratar (años):',
          P.sup_antiguedad.describe()[['mean','50%','min','max']].round(1).to_dict())
    print(f'Filas con contrato ANTERIOR a la constitución de la empresa: {int((P.sup_antiguedad < 0).sum()):,}')

# ---------- 3) Provincia distinta a la de la entidad ----------
cols_bmv = [c[0] for c in con.execute("DESCRIBE bmv").fetchall()]
col_ep = next((c for c in cols_bmv if 'provincia' in c.lower() and 'entidad' in c.lower()), None)
col_ep = col_ep or next((c for c in cols_bmv if 'provincia' in c.lower()), None)
print(f'\nProvincia de la entidad en bmv: {col_ep}')
if col_ep and col_pro:
    con.register('sup_t', sup[['proveedor_ruc','sup_provincia']].dropna())
    mm = con.execute(f"""
        SELECT b.proveedor_ruc, b.anio_proceso,
               AVG(CASE WHEN UPPER(TRIM(b."{col_ep}")) <> s.sup_provincia THEN 1.0 ELSE 0.0 END) AS sup_frac_fuera_provincia
        FROM bmv b JOIN sup_t s ON s.proveedor_ruc = b.proveedor_ruc
        WHERE b."{col_ep}" IS NOT NULL GROUP BY 1,2""").df()
    P = P.merge(mm, on=['proveedor_ruc','anio_proceso'], how='left')
    print('Fracción de contratos fuera de la provincia del proveedor:',
          P.sup_frac_fuera_provincia.describe()[['mean','50%']].round(3).to_dict())

# ---------- 4) ¿Alguna sirve? AUC contra los dos objetivos ----------
def auc(x, y):
    m = ~pd.isna(x)
    if m.sum() < 20 or y[m].sum() == 0 or y[m].sum() == m.sum(): return np.nan
    r = pd.Series(x[m]).rank().values; yy = y[m]
    n1, n0 = yy.sum(), len(yy) - yy.sum()
    return (r[yy == 1].sum() - n1*(n1+1)/2) / (n1*n0)

cand = [c for c in P.columns if c.startswith('sup_') and P[c].dtype.kind in 'fiub']
filas = []
for c in cand:
    x = P[c].astype(float).values
    filas.append((c, auc(x, P.y_fantasma.values), auc(x, P.y_intragrupo.values),
                  float(P[c].notna().mean()),
                  float(np.nanmean(x[P.y_fantasma.values == 1])) if P.y_fantasma.sum() else np.nan,
                  float(np.nanmean(x[P.y_fantasma.values == 0]))))
res = pd.DataFrame(filas, columns=['variable','auc_A','auc_B','cobertura','media_positivos_A','media_resto'])
res['separa'] = np.where(res[['auc_A','auc_B']].sub(0.5).abs().max(axis=1) > 0.10, 'SI', '')
print('\n--- ¿Sirve alguna variable nueva de la Superintendencia? ---')
print(res.round(3).sort_values('auc_A', key=lambda s: s.sub(0.5).abs(), ascending=False).to_string(index=False))

for c in ['sup_situacion','sup_tipo','sup_provincia','sup_cargo']:
    if c in P.columns and P[c].notna().any():
        t = (P.groupby(c)
               .agg(filas=('y_fantasma','size'), pos_A=('y_fantasma','sum'), pos_B=('y_intragrupo','sum'))
               .query('filas >= 200').sort_values('pos_A', ascending=False).head(8))
        t['tasa_A_x1000'] = (1000*t.pos_A/t.filas).round(2); t['tasa_B_x1000'] = (1000*t.pos_B/t.filas).round(2)
        print(f'\n--- {c} ---'); print(t.to_string())

P.to_parquet(f'{R}/panel_con_superintendencia.parquet', index=False)
print(f'\nGuardado panel_con_superintendencia.parquet ({len(P):,} filas x {P.shape[1]} columnas)')
print('Si alguna variable marca SI en la columna "separa", la agregamos al panel definitivo.')

In [ ]:
# ===================================================================
# CUADERNO 7 · CELDA 7 — Incorporar al panel SOLO las variables legítimas
# de la Superintendencia, y dejar fuera las que son consecuencia de la etiqueta.
# ===================================================================
import os, shutil, warnings
import numpy as np, pandas as pd
warnings.filterwarnings('ignore')
R = '03_resultados'

PS = pd.read_parquet(f'{R}/panel_con_superintendencia.parquet')
print(f'Panel con Superintendencia: {len(PS):,} filas x {PS.shape[1]} columnas')

# --- las que NO entran: son el efecto administrativo de la calificación del SRI ---
FUERA = ['sup_no_activa','sup_disolucion','sup_situacion','sup_rep_liquidador']
# --- las que SÍ entran: anteriores al contrato o independientes del registro del SRI ---
DENTRO = ['sup_antiguedad','sup_anio_constitucion','sup_muy_joven',
          'sup_frac_fuera_provincia','sup_contrata_antes_de_existir','sup_tipo','sup_provincia']
print(f'\nSe excluyen (consecuencia de la etiqueta): {FUERA}')
print(f'Se incorporan (antecedentes legítimos):     {DENTRO}')

nuevas = []
for c in DENTRO:
    if c not in PS.columns: print(f'  !! no existe {c}'); continue
    if not pd.api.types.is_numeric_dtype(PS[c]):   # ojo: parquet devuelve string[pyarrow], no object
        top = PS[c].value_counts().head(6).index
        for v in top:
            nom = 'sup_' + str(v).lower().replace(' ','_')[:22]
            PS[nom] = (PS[c] == v).astype(float); nuevas.append(nom)
    else:
        nuevas.append(c)
print(f'\nVariables numéricas resultantes ({len(nuevas)}): {nuevas}')

def auc(x, y):
    m = ~pd.isna(x)
    if m.sum() < 20 or y[m].sum() == 0 or y[m].sum() == m.sum(): return np.nan
    r = pd.Series(x[m]).rank().values; yy = y[m]
    n1, n0 = yy.sum(), len(yy) - yy.sum()
    return (r[yy == 1].sum() - n1*(n1+1)/2) / (n1*n0)
chk = pd.DataFrame([(c, auc(PS[c].astype(float).values, PS.y_fantasma.values),
                        auc(PS[c].astype(float).values, PS.y_intragrupo.values),
                        float(PS[c].notna().mean())) for c in nuevas],
                   columns=['variable','auc_A','auc_B','cobertura'])
print('\n--- Comprobación de las que se incorporan ---')
print(chk.round(3).sort_values('auc_A', key=lambda s: s.sub(0.5).abs(), ascending=False).to_string(index=False))
peligro = chk[(chk.auc_A.sub(0.5).abs() > 0.40) | (chk.auc_B.sub(0.5).abs() > 0.40)]
assert len(peligro) == 0, f'Todavía hay fuga: {list(peligro.variable)}'
print('Ninguna supera el umbral de fuga.')

base = [c for c in PS.columns if not c.startswith('sup_') or c in nuevas]
P2 = PS[base].copy()
if not os.path.exists(f'{R}/panel_modelo_sin_supercias.parquet'):
    shutil.copy2(f'{R}/panel_modelo.parquet', f'{R}/panel_modelo_sin_supercias.parquet')
    print('\nRespaldo: panel_modelo_sin_supercias.parquet')
P2.to_parquet(f'{R}/panel_modelo.tmp.parquet', index=False)
os.replace(f'{R}/panel_modelo.tmp.parquet', f'{R}/panel_modelo.parquet')
print(f'panel_modelo.parquet actualizado: {len(P2):,} filas x {P2.shape[1]} columnas')

print('\n--- Las dos variables nuevas, en números ---')
for c in ['sup_antiguedad','sup_frac_fuera_provincia']:
    if c in P2.columns:
        a = P2.loc[P2.y_fantasma == 1, c].mean(); b = P2.loc[P2.y_fantasma == 0, c].mean()
        d = P2.loc[P2.y_intragrupo == 1, c].mean()
        print(f'  {c:28s} fantasma {a:7.3f} | intragrupo {d:7.3f} | resto {b:7.3f}')
print('\nAhora: volver a correr la Celda 4b y después la Celda 5.')

In [ ]:
# ===================================================================
# CUADERNO 7 · CELDA 4b — Variables del modelo y AUDITORÍA DE FUGA (corregida)
# Reemplaza a la Celda 4. Regla: la fuga se decide por el ORIGEN de la variable,
# no por su AUC. El AUC solo se reporta para revisión humana.
# ===================================================================
import os, json, warnings
import numpy as np, pandas as pd
warnings.filterwarnings('ignore')
R = '03_resultados'
AVISO_AUC = 0.80        # solo para avisar, no excluye

P = pd.read_parquet(f'{R}/panel_modelo.parquet')
print(f'Panel: {len(P):,} filas x {P.shape[1]} columnas')

LLAVES    = {'proveedor_ruc','anio_proceso'}
ETIQUETAS = {'y_fantasma','y_intragrupo','en_catastro','n_proc_intragrupo','tiene_grupo'}
IDS       = {'com_id'}
# (1) variables que nacen del MISMO registro del que sale la etiqueta A (catastro del SRI):
#     una empresa calificada como fantasma tiene el RUC suspendido por definición.
PAT_SRI  = ('fantasma','estado_riesgoso','suspendid','ruc_estado','catastro','disolucion', 'liquidac')
EXC_SRI  = {c for c in P.columns if any(p in c.lower() for p in PAT_SRI)} - ETIQUETAS
# (2) variables definicionales del objetivo B: ser positivo exige estar en el directorio societario
PAT_DIR  = ('sociedad_registrada','en_directorio','tiene_representante')
EXC_DIR  = {c for c in P.columns if any(p in c.lower() for p in PAT_DIR)}

print('\n--- Exclusiones por origen ---')
print(f'  Siempre (identificador):            {sorted(IDS)}')
print(f'  Objetivo A y B (vienen del SRI):    {sorted(EXC_SRI)}')
print(f'  Objetivo B (definición del grupo):  {sorted(EXC_DIR)}')
num = P.select_dtypes(include=[np.number, bool]).columns
X_cols = [c for c in num if c not in LLAVES | ETIQUETAS | IDS | EXC_SRI]
print(f'\nCandidatas: {len(X_cols)}')

def auc(x, y):
    m = ~pd.isna(x)
    if m.sum() < 20 or y[m].sum() == 0 or y[m].sum() == m.sum(): return np.nan
    r = pd.Series(x[m]).rank().values; yy = y[m]
    n1, n0 = yy.sum(), len(yy) - yy.sum()
    return (r[yy == 1].sum() - n1*(n1+1)/2) / (n1*n0)

aud = pd.DataFrame([(c, auc(P[c].astype(float).values, P.y_fantasma.values),
                        auc(P[c].astype(float).values, P.y_intragrupo.values),
                        float(pd.isna(P[c]).mean())) for c in X_cols],
                   columns=['variable','auc_A','auc_B','pct_nulos'])

lim = AVISO_AUC - 0.5
avisa = aud[(aud.auc_A.sub(0.5).abs() > lim) | (aud.auc_B.sub(0.5).abs() > lim)]
print(f'\n--- AVISO: variables con |AUC-0.5| > {lim:.2f} que SÍ entran al modelo ---')
if len(avisa):
    print(avisa.round(3).to_string(index=False))
    print('Revisa una por una de dónde sale cada una. Si alguna se construyó a partir del')
    print('catastro del SRI o del directorio, agrégala a PAT_SRI / PAT_DIR y vuelve a correr.')
else:
    print('  ninguna')

FAM = lambda pre: [c for c in X_cols if c.startswith(pre)]
fam = {'red':FAM('red_'), 'cob':FAM('cob_'), 'com':FAM('com_'),
       'rep':[c for c in X_cols if c.startswith('rep_') or c.startswith('grupo_')],
       'ent':FAM('ent_'), 'idx':FAM('idx_'),
       'cri':[c for c in X_cols if c.startswith('cri2') or c.startswith('dim_')]}
fam['base'] = [c for c in X_cols if c not in sum(fam.values(), [])]

PAT_B     = ('intragrupo',)      # el fenomeno que el objetivo B quiere predecir, rezagado
EXC_B_PAT = {c for c in X_cols if any(p in c.lower() for p in PAT_B)}
exc_B     = set(fam['rep']) | EXC_DIR | EXC_B_PAT
VARS = {'A': X_cols,
        'B': [c for c in X_cols if c not in exc_B]}
ANT = fam['red'] + fam['cob'] + fam['com'] + fam['idx']       # solo años anteriores
VARS['A_anticipada'] = ANT
VARS['B_anticipada'] = [c for c in ANT if c not in exc_B]
alto = set(avisa.variable)                                     # variante estricta, para sensibilidad
VARS['A_estricta'] = [c for c in VARS['A'] if c not in alto]
VARS['B_estricta'] = [c for c in VARS['B'] if c not in alto]
print('\n--- Conjuntos ---')
print(f'  Objetivo B (fenómeno rezagado):     {sorted(EXC_B_PAT)}')
for k, v in VARS.items(): print(f'  {k:14s} {len(v):3d} variables')

for obj, col in [('A','auc_A'), ('B','auc_B')]:
    s = aud[aud.variable.isin(VARS[obj])]
    print(f'\n--- Más informativas que entran al objetivo {obj} ---')
    print(s.reindex(s[col].sub(0.5).abs().sort_values(ascending=False).index).head(10).round(3).to_string(index=False))

print('\n--- Nulos por familia ---')
for f, cs in fam.items():
    if cs: print(f'  {f:5s} {len(cs):3d} vars | nulos promedio {P[cs].isna().mean().mean()*100:5.1f}%')

with open(f'{R}/vars_modelo.json','w',encoding='utf-8') as f: json.dump(VARS, f, ensure_ascii=False, indent=1)
aud.to_csv(f'{R}/auditoria_fuga.csv', index=False)
print('\nGuardado: vars_modelo.json y auditoria_fuga.csv')

In [ ]:
# ===================================================================
# CUADERNO 7 · CELDA 5 — Modelo PU (bagging) y evaluación honesta
# ===================================================================
import os, re, sys, time, json, unicodedata, warnings
import numpy as np, pandas as pd
warnings.filterwarnings('ignore')
try:
    from sklearn.ensemble import HistGradientBoostingClassifier
except ImportError:
    import subprocess; subprocess.run([sys.executable,'-m','pip','install','scikit-learn'], check=True)
    from sklearn.ensemble import HistGradientBoostingClassifier
R = '03_resultados'

K_BAG     = 100      # modelos por conjunto (bagging PU)

RATIO_U   = 5        # no etiquetados por cada positivo en cada modelo
N_POR_ANIO= 2000     # muestra de población por año para calcular percentiles
B_NULL    = 5000     # repeticiones de la hipótesis nula
SEMILLA   = 20260930
rng = np.random.default_rng(SEMILLA)

assert callable(norm_nombre), 'Falta correr la Celda 0 del cuaderno 7'
P = pd.read_parquet(f'{R}/panel_modelo.parquet')
VARS = json.load(open(f'{R}/vars_modelo.json', encoding='utf-8'))

# ---------- conglomerado del objetivo B: el representante legal, no la empresa ----------
dr = pd.read_parquet(f'{R}/directorio_companias.parquet')[['RUC','REPRESENTANTE']].drop_duplicates('RUC')
dr['rep_norm'] = dr.REPRESENTANTE.map(norm_nombre)
P = P.merge(dr[['RUC','rep_norm']], left_on='proveedor_ruc', right_on='RUC', how='left').drop(columns=['RUC'])
P['grupo_B'] = P.rep_norm.fillna('ruc:' + P.proveedor_ruc)


def pu_bagging(Xp, Xu, Xs, k=K_BAG, ratio=RATIO_U, seed=0):
    """Mordelet y Vert: K modelos, cada uno con todos los positivos y una muestra de no etiquetados."""
    r = np.random.default_rng(seed); acum = np.zeros(len(Xs)); n = len(Xp); hechos = 0
    for i in range(k):
        idx = r.choice(len(Xu), size=min(len(Xu), n*ratio), replace=False)
        Xtr = np.vstack([Xp, Xu[idx]])
        ytr = np.r_[np.ones(n), np.zeros(len(idx))]
        # con pocos positivos, en cada submuestra hay columnas con 0 o 1 valor distinto:
        # el binado de sklearn falla con esas, así que se descartan en cada modelo
        util = np.array([np.unique(Xtr[~np.isnan(Xtr[:, j]), j]).size >= 2
                         for j in range(Xtr.shape[1])])
        if util.sum() == 0: continue
        m = HistGradientBoostingClassifier(max_depth=3, max_iter=60, learning_rate=0.1,
                                           min_samples_leaf=5, l2_regularization=1.0,
                                           random_state=i).fit(Xtr[:, util], ytr)
        acum += m.predict_proba(Xs[:, util])[:, 1]; hechos += 1
    assert hechos > 0, 'Ningún modelo se pudo entrenar: revisar las variables'
    return acum / hechos

def evaluar(objetivo, col_y, col_grupo, conjunto, n_folds=None):
    cols = VARS[conjunto]
    d = P[P[col_y].notna()].copy()
    if objetivo == 'B': d = d[d.tiene_grupo]            # universo: solo quien tiene grupo identificable
    X = d[cols].astype(float).values
    y = d[col_y].values.astype(int); g = d[col_grupo].values
    gp = pd.unique(g[y == 1])
    print(f'\n{"="*62}\nObjetivo {objetivo} | conjunto "{conjunto}" | {len(cols)} variables')
    print(f'  filas: {len(d):,} | positivos: {y.sum()} en {len(gp)} conglomerados')

    anios = d.anio_proceso.values
    idx_pos = np.where(y == 1)[0]
    idx_neg = np.concatenate([rng.choice(np.where((y == 0) & (anios == a))[0],
                              size=min(N_POR_ANIO, ((y == 0) & (anios == a)).sum()), replace=False)
                              for a in np.unique(anios) if ((y == 0) & (anios == a)).sum() > 0])
    idx_s = np.concatenate([idx_pos, idx_neg]); Xs = X[idx_s]
    score = np.full(len(idx_s), np.nan)

    assert len(gp) >= 2, (f'El objetivo {objetivo} tiene {len(gp)} conglomerado(s) de positivos: '
                          'no se puede dejar uno fuera. Revisar la definición del conglomerado.')
    folds = ([[c] for c in gp] if n_folds is None
             else [f for f in np.array_split(rng.permutation(gp), min(n_folds, len(gp))) if len(f)])
    t0 = time.time(); base_pob = np.zeros(len(Xs))
    for j, fuera in enumerate(folds):
        m_tr = (y == 1) & (~pd.Series(g).isin(fuera).values)   # positivos SIN el pliegue
        m_u  = (y == 0)
        s = pu_bagging(X[m_tr], X[m_u], Xs, seed=SEMILLA + j)
        donde = np.isin(g[idx_s], fuera) & (y[idx_s] == 1)
        score[donde] = s[donde]
        if j == 0: base_pob = s                                 # modelo del primer pliegue, para la población
    score[np.isnan(score)] = base_pob[np.isnan(score)]
    print(f'  {len(folds)} pliegues entrenados en {time.time()-t0:.0f} s')

    ev = pd.DataFrame({'g': g[idx_s], 'y': y[idx_s], 'anio': anios[idx_s], 'score': score})
    for nombre, v in [('modelo', score),
                      ('cri_v2', d['cri2_medio'].values[idx_s] if 'cri2_medio' in d else None),
                      ('idx_intermediacion', d['idx_intermediacion'].values[idx_s] if 'idx_intermediacion' in d else None)]:
        if v is None: continue
        ev[nombre] = v
    res = []
    for nombre in [c for c in ['modelo','cri_v2','idx_intermediacion'] if c in ev]:
        pct = ev.groupby('anio')[nombre].rank(pct=True)
        obs = pct[ev.y == 1].median()
        gneg = pd.unique(ev.loc[ev.y == 0, 'g'])
        pos_idx = {gg: np.where(ev.g.values == gg)[0] for gg in gneg}
        tam = len(pd.unique(ev.loc[ev.y == 1, 'g']))
        nul = np.empty(B_NULL); pv = pct.values
        for b in range(B_NULL):
            sel = rng.choice(gneg, size=tam, replace=False)
            nul[b] = np.nanmedian(pv[np.concatenate([pos_idx[s_] for s_ in sel])])
        p = (1 + np.sum(np.abs(nul - 0.5) >= abs(obs - 0.5))) / (B_NULL + 1)
        top10 = float((pct[ev.y == 1] > 0.9).mean()); top25 = float((pct[ev.y == 1] > 0.75).mean())
        res.append((nombre, round(obs,3), round(np.percentile(nul,2.5),3), round(np.percentile(nul,97.5),3),
                    round(p,4), round(top10,3), round(top25,3), round(top10/0.10,1)))
    r = pd.DataFrame(res, columns=['metodo','mediana_percentil','nulo_p2.5','nulo_p97.5','p',
                                   'frac_decil_sup','frac_cuartil_sup','lift_decil'])
    print(r.to_string(index=False))
    return r, ev

rA, evA = evaluar('A', 'y_fantasma',  'proveedor_ruc', 'A')              # dejar una empresa fuera
rB, evB = evaluar('B', 'y_intragrupo','grupo_B',       'B', n_folds=5)   # validación cruzada por grupos
rAa, _  = evaluar('A', 'y_fantasma',  'proveedor_ruc', 'A_anticipada')   # solo variables de años previos
rBa, _  = evaluar('B', 'y_intragrupo','grupo_B',       'B_anticipada', n_folds=5)

out = pd.concat([rA.assign(caso='A completo'), rAa.assign(caso='A anticipada'),
                 rB.assign(caso='B completo'), rBa.assign(caso='B anticipada')])
out.to_csv(f'{R}/resultados_modelo.csv', index=False)
print('\n\n=============== RESUMEN ===============')
print(out[['caso','metodo','mediana_percentil','nulo_p2.5','nulo_p97.5','p','lift_decil']].to_string(index=False))
print('\nGuardado resultados_modelo.csv')
print('lift_decil = cuántas veces más positivos caen en el decil superior que por azar (1.0 = azar)')